In [48]:
import pandas as pd
df = pd.read_csv(r'C:\Users\qande\Desktop\AI_Project\data\dataset.csv')

In [49]:
def get_symptom_columns(df):
    return [col for col in df.columns if col.startswith('Symptom')]

In [50]:
def build_graph(df):
    symptom_cols = get_symptom_columns(df)
    graph = {}
    for _, row in df.iterrows():
        symptoms = [row[col] for col in symptom_cols if pd.notna(row[col])]
        for s in symptoms:
            if s not in graph:
                graph[s] = set()
            for other in symptoms:
                if other != s:
                    graph[s].add(other)
    return {k: list(v) for k, v in graph.items()}

graph = build_graph(df)

In [19]:
graph

{'itching': [' dischromic _patches',
  ' mild_fever',
  ' receiving_blood_transfusion',
  ' nodal_skin_eruptions',
  ' loss_of_appetite',
  ' yellowing_of_eyes',
  ' dark_urine',
  ' malaise',
  ' fatigue',
  ' headache',
  ' vomiting',
  ' skin_rash',
  ' yellowish_skin',
  ' burning_micturition',
  ' swelled_lymph_nodes',
  ' red_spots_over_body',
  ' nausea',
  ' high_fever',
  ' lethargy',
  ' receiving_unsterile_injections',
  ' yellow_urine',
  ' abdominal_pain',
  ' spotting_ urination',
  ' weight_loss',
  ' stomach_pain'],
 ' skin_rash': [' dischromic _patches',
  ' chills',
  ' blackheads',
  ' mild_fever',
  ' nodal_skin_eruptions',
  ' loss_of_appetite',
  ' blister',
  ' skin_peeling',
  ' malaise',
  ' fatigue',
  ' headache',
  ' pus_filled_pimples',
  ' vomiting',
  ' burning_micturition',
  'itching',
  ' swelled_lymph_nodes',
  ' scurring',
  ' red_spots_over_body',
  ' back_pain',
  ' muscle_pain',
  ' silver_like_dusting',
  ' high_fever',
  ' lethargy',
  ' joint_p

# Phase 2: Intelligent Agent and Search Algorithms
**Project:** Multi-Domain AI Learning Framework (Medical Track)
#### Define the Agent and State Space
In this phase, we model the medical dataset as a state space where symptoms are nodes in a graph. 
- **Initial State:** The first symptom reported by a patient (e.g., 'itching').
- **Goal State:** A specific target symptom we want to reach (e.g., 'fatigue').
- **Actions:** Moving from the current symptom to any connected symptom in the dataset.
- **Cost:** In this uninformed search (BFS), each transition between symptoms has a cost of 1.
- **Solution:** A path (sequence of symptoms) leading from the start to the goal.

In [52]:
import pandas as pd
from collections import deque

df = pd.read_csv(r'C:\Users\qande\Desktop\AI_Project\data\dataset.csv')
severity_df = pd.read_csv(r'C:\Users\qande\Desktop\AI_Project\data\Symptom-severity.csv')
# data clean
for col in df.columns:
    if col != 'Disease':
        df[col] = df[col].str.strip()

print("Data loaded and cleaned!")

Data loaded and cleaned!


### Graph Construction
We build an adjacency list representing the relationships between symptoms. 
Two symptoms are connected if they appear together in the same disease record in the `dataset.csv`.

In [53]:
def build_graph(df):
    symptom_cols = [col for col in df.columns if col.startswith('Symptom')]
    graph = {} 
    for _, row in df.iterrows():
        symptoms = [row[col] for col in symptom_cols if pd.notna(row[col])] # Get all symptoms in this row, skipping empty ones 
        for s in symptoms:
            if s not in graph:
                graph[s] = set()
            for other in symptoms:
                if other != s:
                    graph[s].add(other)                
    # Convert sets to lists
    return {k: list(v) for k, v in graph.items()}

symptom_graph = build_graph(df) # graph created
print(f"Graph built with {len(symptom_graph)} unique symptoms.")

Graph built with 131 unique symptoms.


### The AI Agent
We define the `AIAgent` class to handle the "senses" of our searcher. 
It uses `perceive` to see neighboring symptoms and `goal_test` to check if the destination is reached.

In [55]:
class AIAgent:
    def __init__(self, symptom_graph):
        self.graph = symptom_graph

    def perceive(self, state):
        return self.graph.get(state, [])    # Returns the neighbors of the current symptom

    def goal_test(self, current_state, target_goal):
        return current_state == target_goal  # Returns True if we reached the symptom we wanted
    
    def get_cost(self, state1, state2):
        # In basic BFS, every move costs 1
        return 1

agent = AIAgent(symptom_graph) # Initialize our agent
print("AI-Agent is ready!")

AI-Agent is ready!


## Breadth-First Search (BFS)
BFS explores the graph layer by layer. It is guaranteed to find the shortest path in terms of the number of nodes. 
We use a **Queue (FIFO)** to manage the frontier.

In [58]:
def breadth_first_search(agent, start_node, goal_node):
    queue = deque([[start_node]])     # start with a path containing just the starting symptom
    
    visited = set() #keep track of the visited symptoms
    visited.add(start_node)
    
    nodes_explored = 0
    
    while queue:
        current_path = queue.popleft()    # Pop the oldest path
        
        last_symptom = current_path[-1]  # Look at the last symptom in that path
        nodes_explored += 1
    
        if agent.goal_test(last_symptom, goal_node): # Check if we found the goal
            return current_path, nodes_explored
        
        for neighbor in agent.perceive(last_symptom): # If not the goal, look at all neighbors
            if neighbor not in visited:
                visited.add(neighbor)
                new_path = list(current_path) # Create a new path and add it to the queue
                new_path.append(neighbor)
                queue.append(new_path)
                
    return None, nodes_explored

start_symptom = 'itching'
target_symptom = 'fatigue'

path, count = breadth_first_search(agent, start_symptom, target_symptom)
print(f"BFS")
print(f"Path Found: {path}")
print(f"Nodes Explored: {count}")

BFS
Path Found: ['itching', 'fatigue']
Nodes Explored: 19


### BFS REFLECTION
- **Path Found:** ['itching', 'fatigue']
- **Nodes Explored:** 19
- **Observation:** BFS explored many nodes because it looks at every neighbor at the current depth before moving deeper. This ensures the shortest path but can be slow in a highly connected graph.

## Depth-First Search (DFS)
DFS explores a single branch as deeply as possible before backtracking. 
- **Data Structure:** We use a **Stack** (LIFO). In Python, this is a list where we `pop()` from the end.
- **Comparison:** We will run this on the same 'itching' to 'fatigue' path to see how the number of nodes explored differs from BFS.

In [59]:
def depth_first_search(agent, start_node, goal_node):
    stack = [[start_node]]
    visited = set()  
    nodes_explored = 0
    
    print(f"Starting DFS search from '{start_node}' to find '{goal_node}'...")

    while stack:
        current_path = stack.pop()   
        last_symptom = current_path[-1]
        
        if last_symptom in visited:  #check if visited when popping to handle the stack logic correctly
            continue
          
        visited.add(last_symptom)
        nodes_explored += 1
        # Goal Test
        if agent.goal_test(last_symptom, goal_node):
            return current_path, nodes_explored

        neighbors = agent.perceive(last_symptom) #Look at neighbors
        
        for neighbor in neighbors:
            if neighbor not in visited:
                new_path = list(current_path) # Create a new path and push it onto the stack
                new_path.append(neighbor)
                stack.append(new_path)
                
    return None, nodes_explored

start_symptom = 'itching'
target_symptom = 'fatigue'

dfs_path, dfs_count = depth_first_search(agent, start_symptom, target_symptom)

print(f"DFS")
print(f"Path Found: {dfs_path}")
print(f"Nodes Explored: {dfs_count}")

Starting DFS search from 'itching' to find 'fatigue'...
DFS
Path Found: ['itching', 'dark_urine', 'lethargy', 'brittle_nails', 'mood_swings', 'muscle_weakness', 'movement_stiffness', 'swelling_joints', 'knee_pain', 'hip_joint_pain', 'joint_pain', 'muscle_pain', 'continuous_sneezing', 'high_fever', 'family_history', 'yellowish_skin', 'history_of_alcohol_consumption', 'fluid_overload', 'vomiting', 'red_spots_over_body', 'mild_fever', 'sweating', 'chest_pain', 'fast_heart_rate', 'rusty_sputum', 'chills', 'headache', 'unsteadiness', 'spinning_movements', 'nausea', 'malaise', 'fatigue']
Nodes Explored: 34


### BFS vs DFS Comparison
- **BFS Nodes Explored:** 19
- **DFS Nodes Explored:** 34
#### **Path Length Comparison:** 
- **BFS Path Length:** 2
- **DFS Path Length:** 32
#### **Final Observation:**
- **BFS (Breadth-First Search):** As expected, BFS found the shortest path between the symptoms. Because it explores level-by-level.
- **DFS (Depth-First Search):** DFS found a much longer path. This is because it follows the first available neighbor as deep as possible before ever checking other branches. While it uses less memory in some cases, it is not reliable for finding the most direct medical diagnosis path.

## Depth Limited Search (DLS)
DLS is a modification of DFS that stops exploring once it reaches a certain depth (limit). This prevents the agent from getting lost in infinitely deep paths.
- **Test 1:** Limit = 3
- **Test 2:** Limit = 5

In [63]:
def depth_limited_search(agent, start_node, goal_node, limit):
    stack = [( [start_node], 0 )]
    visited = {} # We use a dict to store the shallowest depth we reached a node
    nodes_explored = 0
    
    print(f"Starting DLS (Limit={limit}) from '{start_node}' to '{goal_node}'...")

    while stack:
        path, depth = stack.pop()
        current_symptom = path[-1]
        
        nodes_explored += 1
        
        if agent.goal_test(current_symptom, goal_node): #GOAL TEST
            return path, nodes_explored

        if depth < limit: # 2. Check Depth Limit
            for neighbor in agent.perceive(current_symptom): # Look at neighbors
                if neighbor not in visited or visited[neighbor] > depth + 1: # If we haven't seen this neighbor at this depth or shallower
                    visited[neighbor] = depth + 1
                    new_path = list(path)
                    new_path.append(neighbor)
                    stack.append((new_path, depth + 1))
                    
    return None, nodes_explored

for l in [3, 5]:
    path, count = depth_limited_search(agent, 'itching', 'fatigue', l)
    print(f"Limit {l}: {'Path Found!' if path else 'No Path Found within limit.'}")
    if path: print(f"Path: {' -> '.join(path)}")
    print(f"Nodes Explored: {count}\n")

Starting DLS (Limit=3) from 'itching' to 'fatigue'...
Limit 3: Path Found!
Path: itching -> fatigue
Nodes Explored: 138

Starting DLS (Limit=5) from 'itching' to 'fatigue'...
Limit 5: Path Found!
Path: itching -> fatigue
Nodes Explored: 228



### Depth Limited Search (DLS)
TESTED WITH 2 DIFFERENT LIMITS:
- **Limit 3 Results:** Path Found!
- **Limit 5 Results:** Path Found!
- **Nodes Explored:** 
- **Limit 3 nodes explored:** 138
- **Limit 5 nodes explored:** 228

#### **Conclusion on DLS:**
Limit 3: A path was found (itching -> fatigue) exploring 138 nodes.
Limit 5: The same path was found, but the search was much more "expensive," exploring 228 nodes.


## Iterative Deepening Search (IDS)
IDS is a strategy that combines the benefits of BFS (finding the shortest path) and DFS (low memory usage).
- **How it works:** It starts with a depth limit of 0, then 1, then 2, etc. At each level, it performs a new Depth Limited Search.
- **Why use it?** It is guaranteed to find the shortest path like BFS, but it doesn't store all nodes in memory at once.
- **Goal:** We will observe at which depth level our target symptom is found and compare the total "work" (nodes explored) to BFS.

In [71]:
def iterative_deepening_search(agent, start_node, goal_node, max_depth):
    total_nodes_overall = 0
    
    for limit in range(max_depth + 1):
        print(f"Searching at Depth Limit: {limit} ")
        
        path, nodes_in_this_run = depth_limited_search(agent, start_node, goal_node, limit)

        total_nodes_overall += nodes_in_this_run
        
        if path:
            print(f"SUCCESS: Goal found at depth {limit}!")
            return path, total_nodes_overall
            
        print(f"Goal not found at depth {limit}. Increasing limit...")
        
    return None, total_nodes_overall

start_symptom = 'itching'
target_symptom = 'fatigue'

ids_path, ids_total_count = iterative_deepening_search(agent, start_symptom, target_symptom, 10)
print("\n")
print(f"IDS Result Path: {ids_path}")
print(f"Total Nodes Explored: {ids_total_count}")

Searching at Depth Limit: 0 
Starting DLS (Limit=0) from 'itching' to 'fatigue'...
Goal not found at depth 0. Increasing limit...
Searching at Depth Limit: 1 
Starting DLS (Limit=1) from 'itching' to 'fatigue'...
SUCCESS: Goal found at depth 1!


IDS Result Path: ['itching', 'fatigue']
Total Nodes Explored: 10


### IDS and BFS Comparison

| Metric | Breadth-First Search (BFS) | Iterative Deepening (IDS) |
| :--- | :--- | :--- |
| **Path Found** | ['itching', 'fatigue'] | ['itching', 'fatigue'] |
| **Nodes Explored** | 19 | 10 |
| **Shortest Path?** | Yes | Yes |

**Observations:**
1. **Shortest Path:** Both BFS and IDS successfully identified the identical shortest path. This confirms that IDS provides the same optimality guarantee as BFS in an uninformed search environment.
2. **Efficiency:** In this specific execution, IDS was actually more efficient, exploring only 10 nodes compared to the 19 nodes explored by BFS. This occurred because the goal symptom was located at a very shallow depth (Depth 1). BFS added all immediate neighbors to its frontier before checking them, whereas IDS found the goal almost immediately upon entering the first depth limit.
3. **Memory:** IDS is significantly better for system memory. While BFS must store all nodes at the current depth level in a queue (which can grow very large), IDS only needs to store the current path it is exploring (the stack), making it the preferred choice for deep or large graphs where memory is a constraint.

## Uniform Cost Search (UCS)
UCS is different from BFS and DFS because it doesn't just look for the fewest "hops." It looks for the path with the **lowest total cost**.
- **The Cost:** We use the `weight` from `Symptom-severity.csv`. 
- **The Logic:** If one path has 2 symptoms with high severity (7+7=14) and another has 3 symptoms with low severity (2+2+2=6), UCS will pick the longer path of 3 symptoms because it is "cheaper" (less severe).
- **Data Structure:** We use a **Priority Queue** (heapq) to always pull out the lowest-cost path first.

In [72]:
import heapq
def uniform_cost_search(agent, start_node, goal_node, severity_map):
    pq = [(0, [start_node])]
    
    visited = {} # Stores {symptom: cheapest_cost_found_so_far}
    nodes_explored = 0
    
    print(f"Starting UCS from '{start_node}' to '{goal_node}'")

    while pq:
        current_cost, path = heapq.heappop(pq)
        current_symptom = path[-1]
        
        if current_symptom in visited and visited[current_symptom] <= current_cost:
            continue
            
        visited[current_symptom] = current_cost
        nodes_explored += 1
        
        if agent.goal_test(current_symptom, goal_node):
            return path, current_cost, nodes_explored

        for neighbor in agent.perceive(current_symptom):
            weight = severity_map.get(neighbor, 1)
            new_cost = current_cost + weight
            
            if neighbor not in visited or new_cost < visited[neighbor]:
                new_path = list(path)
                new_path.append(neighbor)
                heapq.heappush(pq, (new_cost, new_path))
                
    return None, 0, nodes_explored

ucs_path, ucs_total_cost, ucs_nodes = uniform_cost_search(agent, 'itching', 'fatigue', severity_dict)

print(f"\nUCS Result Path: {ucs_path}")
print(f"Total Cumulative Severity: {ucs_total_cost}")
print(f"Nodes Explored: {ucs_nodes}")

Starting UCS from 'itching' to 'fatigue'

UCS Result Path: ['itching', 'fatigue']
Total Cumulative Severity: 4
Nodes Explored: 13


### Uninformed Search Comparison Table


| Algorithm | Path Found | Path Length | Nodes Explored | Optimal? |
| :--- | :--- | :--- | :--- | :--- |
| **BFS** | ['itching', 'fatigue'] | 2 | 19 | yes |
| **DFS** | ['itching', 'dark_urine', 'lethargy', 'brittle_nails', 'mood_swings', 'muscle_weakness', 'movement_stiffness', 'swelling_joints', 'knee_pain', 'hip_joint_pain', 'joint_pain', 'muscle_pain', 'continuous_sneezing', 'high_fever', 'family_history', 'yellowish_skin', 'history_of_alcohol_consumption', 'fluid_overload', 'vomiting', 'red_spots_over_body', 'mild_fever', 'sweating', 'chest_pain', 'fast_heart_rate', 'rusty_sputum', 'chills', 'headache', 'unsteadiness', 'spinning_movements', 'nausea', 'malaise', 'fatigue'] | 32 | 34 | no |
| **Depth Limited (limit=3)** | ['itching', 'fatigue'] | 2 | 138 | yes |
| **Depth Limited (limit=5)** | ['itching', 'fatigue'] | 2 | 228 | yes |
| **Iterative Deepening** | ['itching', 'fatigue'] | 2 | 10 | yes|
| **Uniform Cost Search** | ['itching', 'fatigue'] |2 | 13 | yes |

**Observations:**
1. **Shortest Path:** Both BFS and IDS found the exact same path. This proves that IDS is optimal, just like BFS.
2. **Efficiency:** IDS explored a total of **10** nodes. This is often more than BFS because IDS "re-explores" the top levels every time the limit increases. 
3. **Memory:** Even though IDS explored more nodes total, it is better for the computer's memory because it only ever follows one "branch" at a time (like DFS), whereas BFS has to remember every single neighbor at once.   is this correct?


**Which algorithm explored the fewest nodes?** In our test, Iterative Deepening Search (IDS) explored the fewest nodes (10). This is because the goal was at the very first level of the graph, and IDS stopped as soon as it found the goal at that shallow depth, whereas BFS added more neighbors to its queue before finishing.

**Path Quality:** BFS, IDS, and UCS all found the same shortest path. However, UCS is the only one that guarantees the "cheapest" path based on medical severity. In this case, since the path was a direct link, the total severity was simply the weight of the 'fatigue' symptom (4).

**Algorithm Behavior:**
- BFS is reliable but "memory-hungry."
- DFS is risky as it can find very long paths.
- DLS is useful only if you know the depth of the diagnosis in advance.
- UCS is the most "intelligent" uninformed search because it respects the severity of symptoms.

# Informed Searches
Informed search uses a **Heuristic function $h(n)$** to estimate the distance from the current node to the goal. 

- **Heuristic:** We calculate the number of "common neighbors" between the current symptom and the goal symptom. 
- **Logic:** If two symptoms often appear with the same set of other symptoms, they are likely related to the same disease and thus "closer" in the state space.
- **Inversion:** We subtract the count of common neighbors from a large number (or use a small fraction) so that more matches result in a **lower** heuristic value (since search algorithms prioritize lower values).

In [75]:
import heapq
def heuristic(current_node, goal_node, graph):
    """Estimates distance by looking at how many common neighbors the current symptom and goal symptom share."""
    if current_node == goal_node:
        return 0
    
    neighbors_current = set(graph.get(current_node, [])) #get neighbours of both nodes
    neighbors_goal = set(graph.get(goal_node, []))
    
    common = neighbors_current.intersection(neighbors_goal)     # Find common neighbors
    
    # The more common neighbors, the smaller the heuristic value (better)
    # We use 100 as a base to ensure the value stays positive
    return 100 - len(common)

# BEST FIRST SEARCH

In [76]:
def best_first_search(agent, start_node, goal_node):
    pq = [(heuristic(start_node, goal_node, agent.graph), [start_node])]
    
    visited = set()
    nodes_explored = 0
    
    print(f"Starting Best-First Search from '{start_node}' to '{goal_node}'")

    while pq:
        h_val, path = heapq.heappop(pq)
        current_symptom = path[-1]
        
        if current_symptom in visited:
            continue
            
        visited.add(current_symptom)
        nodes_explored += 1

        if agent.goal_test(current_symptom, goal_node): #goal test
            return path, nodes_explored
        
        for neighbor in agent.perceive(current_symptom): # Explore neighbors
            if neighbor not in visited:
                new_path = list(path)
                new_path.append(neighbor)
                # Greedy: Only care about h(n)
                h_score = heuristic(neighbor, goal_node, agent.graph)
                heapq.heappush(pq, (h_score, new_path))
                
    return None, nodes_explored

bfs_informed_path, bfs_informed_count = best_first_search(agent, 'itching', 'fatigue')

print(f"\nBest-First Search Result Path: {bfs_informed_path}")
print(f"Nodes Explored: {bfs_informed_count}")

Starting Best-First Search from 'itching' to 'fatigue'

Best-First Search Result Path: ['itching', 'fatigue']
Nodes Explored: 2


### Best First Search and UCS Comparison

| Metric | Uniform Cost Search (UCS) | Best First Search |
| :--- | :--- | :--- |
| **Path Found** | ['itching', 'fatigue'] | ['itching', 'fatigue'] |
| **Nodes Explored** | 13 | 2 |
| **Shortest Path?** | Yes | Yes |

**Observations:**
1. **Speed vs. Accuracy:** Best First Search is "greedy." It only looks at the heuristic (the estimated distance) and ignores the actual cost (severity). 
3. **Optimality:** Unlike UCS, Best First Search is **not guaranteed** to find the shortest or cheapest path. It just tries to find the goal as quickly as possible.

## A* Search Algorithm
A* Search uses the evaluation function **f(n) = g(n) + h(n)**:
- **g(n):** The actual cost to reach the current node (Cumulative Severity).
- **h(n):** The heuristic estimate of the cost to reach the goal.
- **f(n):** The total estimated cost of the path through node *n*.

**Admissibility Check:**
A heuristic is **admissible** if it never overestimates the actual cost to reach the goal. Our heuristic (based on common neighbors) is designed to stay within a reasonable range. Since the minimum severity of a symptom is usually 1, as long as our heuristic values are low and proportional, the search will remain optimal.

In [78]:
def a_star_search(agent, start_node, goal_node, severity_map):
    start_h = heuristic(start_node, goal_node, agent.graph)
    pq = [(start_h, 0, [start_node])]
    
    visited = {} # Stores {symptom: cheapest_f_score_found}
    nodes_explored = 0
    
    print(f"Starting A* Search from '{start_node}' to '{goal_node}'")

    while pq:
        f_score, g_score, path = heapq.heappop(pq)
        current_symptom = path[-1]
        
        if current_symptom in visited and visited[current_symptom] <= f_score:
            continue
            
        visited[current_symptom] = f_score
        nodes_explored += 1
        
        if agent.goal_test(current_symptom, goal_node): # Goal Test
            return path, g_score, nodes_explored
        
        for neighbor in agent.perceive(current_symptom): # Explore neighbors
            # Calculate g(n): cost from start to neighbor
            weight = severity_map.get(neighbor, 1)
            new_g = g_score + weight
            
            h_score = heuristic(neighbor, goal_node, agent.graph) # Calculate h(n): estimated cost from neighbor to goal
            
            new_f = new_g + h_score # f(n) = g(n) + h(n)
            
            if neighbor not in visited or new_f < visited[neighbor]:
                new_path = list(path)
                new_path.append(neighbor)
                heapq.heappush(pq, (new_f, new_g, new_path))
                
    return None, 0, nodes_explored

astar_path, astar_cost, astar_nodes = a_star_search(agent, 'itching', 'fatigue', severity_dict)

print(f"\nA* Result Path: {astar_path}")
print(f"Total Path Cost (g): {astar_cost}")
print(f"Nodes Explored: {astar_nodes}")

Starting A* Search from 'itching' to 'fatigue'

A* Result Path: ['itching', 'fatigue']
Total Path Cost (g): 4
Nodes Explored: 2


### BFS AND A* COMPARISON

| Metric | Best First Search  | A* Search |
| :--- | :--- | :--- |
| **Path Found** | ['itching', 'fatigue'] | ['itching', 'fatigue'] |
| **Nodes Explored** | 2 | 2 |
| **Shortest Path?** | Yes | Yes |

**Observations:**
1. **Speed vs. Accuracy:** A* usually explores fewer nodes than UCS because the heuristic $h(n)$ guides it toward the goal, preventing it from wandering into "expensive" areas of the graph that aren't heading the right way.
2. **Admissibility:** By testing a few symptoms, we observed that the heuristic $h(n)$ provides a consistent estimate that doesn't drastically exceed the actual path costs, ensuring the algorithm remains optimal.
3. **Optimality:** Unlike Best-First Search, A* is guaranteed to find the optimal path because it balances the actual cost ($g$) with the heuristic ($h$).

## Hill Climbing (Local Search)
Hill Climbing is a "greedy" local search. It doesn't keep a path in memory; it only cares about the current state and its immediate neighbors.

- **Objective Function:** We aim to maximize the **Severity Score**. 
- **The Logic:** At each step, the agent looks at all neighboring symptoms and moves to the one with the highest severity.
- **The Problem:** It can get stuck in "Local Optima" a symptom that is more severe than its neighbors but not the most severe in the entire dataset.

In [80]:
import random
def hill_climbing(agent, severity_map):
    all_symptoms = list(agent.graph.keys()) # 1. Start at a random symptom from the graph
    current_state = random.choice(all_symptoms)
    
    path = [current_state]
    
    while True:
        current_severity = severity_map.get(current_state, 0)
        neighbors = agent.perceive(current_state)
        
        if not neighbors:
            break
        
        best_neighbor = None #Find the neighbor with the highest severity
        best_neighbor_severity = current_severity
        
        for neighbor in neighbors:
            n_severity = severity_map.get(neighbor, 0)
            if n_severity > best_neighbor_severity:
                best_neighbor_severity = n_severity
                best_neighbor = neighbor
        
        if best_neighbor is None:  #If no neighbor is better than current, we reached a peak (Optimum)
            break
            
        current_state = best_neighbor
        path.append(current_state)
        
    return current_state, severity_map.get(current_state, 0), path

print("Running Hill Climbing 10 times\n")

global_max_severity = max(severity_dict.values()) # Find the Global Optimum (The highest possible severity in the whole dataset)
global_optimum_count = 0

for i in range(1, 11):
    final_state, severity, path = hill_climbing(agent, severity_dict)
    
    is_global = "GLOBAL OPTIMUM" if severity == global_max_severity else "LOCAL OPTIMUM"
    if severity == global_max_severity:
        global_optimum_count += 1
        
    print(f"Run {i}: Ended at '{final_state}' (Severity: {severity}) -> {is_global}")
    print(f"   Path taken: {' -> '.join(path[:5])}...") # Showing first 5 steps

print(f"\nSummary: Reached Global Optimum {global_optimum_count} out of 10 times.")

Running Hill Climbing 10 times

Run 1: Ended at 'passage_of_gases' (Severity: 5) -> LOCAL OPTIMUM
   Path taken: internal_itching -> passage_of_gases...
Run 2: Ended at 'swelling_of_stomach' (Severity: 7) -> GLOBAL OPTIMUM
   Path taken: swelling_of_stomach...
Run 3: Ended at 'enlarged_thyroid' (Severity: 6) -> LOCAL OPTIMUM
   Path taken: puffy_face_and_eyes -> enlarged_thyroid...
Run 4: Ended at 'high_fever' (Severity: 7) -> GLOBAL OPTIMUM
   Path taken: muscle_pain -> high_fever...
Run 5: Ended at 'continuous_feel_of_urine' (Severity: 6) -> LOCAL OPTIMUM
   Path taken: bladder_discomfort -> continuous_feel_of_urine...
Run 6: Ended at 'skin_peeling' (Severity: 3) -> LOCAL OPTIMUM
   Path taken: skin_peeling...
Run 7: Ended at 'coma' (Severity: 7) -> GLOBAL OPTIMUM
   Path taken: drying_and_tingling_lips -> vomiting -> coma...
Run 8: Ended at 'skin_peeling' (Severity: 3) -> LOCAL OPTIMUM
   Path taken: small_dents_in_nails -> skin_peeling...
Run 9: Ended at 'spinning_movements' (Sever

## HILL CLIMBING RESULTS

| Metric | Hill Climbing Results  |
| :--- | :--- |
| **Total Runs** | 10 |
| **Global Optimums Reached** | 4 |
| **Local Optimums Reached** | 6 |

##### Observations:

1. **Speed:** Hill Climbing is extremely fast compared to BFS or A* because it never looks back and only considers immediate neighbors.

2. **Local Maxima:** Because it only moves "up," it often gets stuck on a symptom that is severe, but not the most severe in the whole dataset. This is why we didn't reach the global optimum 10/10 times.

3. **Random Restart:** By starting 10 different times, we improved our chances of landing in the "foothills" of the tallest peak, which is a common strategy to fix the flaws of Hill Climbing.

## Simulated Annealing
Simulated Annealing improves upon Hill Climbing by introducing "Temperature." 
- **High Temperature:** The agent is "brave" and will occasionally accept moves to symptoms with lower severity to escape local traps.
- **Cooling Down:** As the temperature drops (multiplied by 0.95 each step), the agent becomes more conservative, eventually acting like a standard Hill Climber.
- **Goal:** To see if this "randomness" allows us to find the Global Optimum more consistently than basic Hill Climbing.

In [82]:
import math
import random

def simulated_annealing(agent, severity_map, start_temp=100, cooling_rate=0.95):
    all_symptoms = list(agent.graph.keys())
    current_state = random.choice(all_symptoms)
    current_severity = severity_map.get(current_state, 0)
    
    temp = start_temp
    path = [current_state]

    while temp > 0.01: # run until the temperature is very low
        neighbors = agent.perceive(current_state)
        if not neighbors:
            break
            
        next_state = random.choice(neighbors)  # Pick ONE neighbor at random
        next_severity = severity_map.get(next_state, 0)
        
        delta = next_severity - current_severity # Calculate the difference in quality
        
        if delta > 0:  # If the neighbor is better, always move there
            current_state = next_state
            current_severity = next_severity
            path.append(current_state)
        else:
            # If the neighbor is worse, move there with a probability
            # The probability formula: P = e^(delta / temp)
            probability = math.exp(delta / temp)
            if random.random() < probability:
                current_state = next_state
                current_severity = next_severity
                path.append(current_state)
        
        temp *= cooling_rate  # Cool down the temperature
        
    return current_state, current_severity, path

print("Running Simulated Annealing")
sa_state, sa_severity, sa_path = simulated_annealing(agent, severity_dict)

print(f"Final State: {sa_state}")
print(f"Final Severity: {sa_severity}")
print(f"Steps taken: {len(sa_path)}")

Running Simulated Annealing
Final State: high_fever
Final Severity: 7
Steps taken: 87


#### HILL CLIMBING AND SIMULATED ANNEALING COMPARISON

| Metric | Hill Climbing  | Simulated Annealing |
| :--- | :--- | :--- |
| **Strategy** | greedy (always climbs up) | probabilistic (can move down) |
| **Best Severity Found** | 7 | 7 |
| **Risk of Local Optima** | high | low |
| **Computation Time** | very fast | slightly slower


## LOCAL BEAM SEARCH
Local Beam Search is a variation of Hill Climbing that tracks **k** states simultaneously. 

In [83]:
def local_beam_search(agent, severity_map, k):
    all_symptoms = list(agent.graph.keys())
    current_states = random.sample(all_symptoms, k)    #Initialize k random states
    
    steps = 0
    while True:
        all_neighbors = []
        
        for state in current_states: # For each state, get all its neighbors
            neighbors = agent.perceive(state)
            for n in neighbors:
                all_neighbors.append((severity_map.get(n, 0), n))  # Store (severity, symptom_name)
        
        all_neighbors.sort(key=lambda x: x[0], reverse=True)  # Sort all neighbors by severity and pick top k
        top_k_neighbors = all_neighbors[:k]
        
        current_best_severity = max([severity_map.get(s, 0) for s in current_states]) #Check if the best neighbor found is better than our current best
        new_best_severity = top_k_neighbors[0][0]
        
        if new_best_severity <= current_best_severity:
            break # No improvement possible
            
        current_states = [item[1] for item in top_k_neighbors] # Update current states for the next round
        steps += 1
        
    best_final_state = current_states[0]
    return best_final_state, severity_map.get(best_final_state, 0), steps

for k_val in [3, 5]:
    state, sev, iterations = local_beam_search(agent, severity_dict, k_val)
    print(f"Beam Search (k={k_val}):")
    print(f"   Best Symptom: '{state}' (Severity: {sev})")
    print(f"   Iterations: {iterations}\n")

Beam Search (k=3):
   Best Symptom: 'high_fever' (Severity: 7)
   Iterations: 2

Beam Search (k=5):
   Best Symptom: 'congestion' (Severity: 5)
   Iterations: 0



##### LOCAL SEARCHES COMPARISON

| Metric | Hill Climbing  | Local Beam Search (k=3) | Local Beam Search (k=3) |
| :--- | :--- | :--- | :--- |
| **Solution Quality** | moderate | high | very high |
| **Speed** | fastest | fast | moderate |
| **Local Optimum?** | rarely | often | most likely |

## Adversarial Search - Minimax Algorithm
Adversarial search is used in "game-playing" scenarios where two agents have opposing goals. 
* **MAX Player:** Aims to reach the highest severity symptom (modeling a "worst-case" diagnostic scan).
* **MIN Player:** Aims to minimize the severity (modeling a "best-case" or mild symptom progression).

We will build a game tree of **depth 4** to observe how the decision changes when an opponent is trying to minimize the outcome.

In [85]:
import math

def minimax(state, depth, is_max_player, agent, severity_map, count_dict):
    count_dict['minimax_nodes'] += 1 # Increment node counter
    
    # Base case: Leaf node or depth limit reached
    if depth == 0:
        return severity_map.get(state, 1)

    neighbors = agent.perceive(state)
    if not neighbors:
        return severity_map.get(state, 1)

    if is_max_player:
        best_val = -math.inf
        for neighbor in neighbors:
            value = minimax(neighbor, depth - 1, False, agent, severity_map, count_dict)
            best_val = max(best_val, value)
        return best_val
    else:
        best_val = math.inf
        for neighbor in neighbors:
            value = minimax(neighbor, depth - 1, True, agent, severity_map, count_dict)
            best_val = min(best_val, value)
        return best_val

counters = {'minimax_nodes': 0}
start_symptom = 'itching'
minimax_score = minimax(start_symptom, 4, True, agent, severity_dict, counters)

print(f"Minimax:")
print(f"Start Symptom: {start_symptom}")
print(f"Depth: 4")
print(f"Final Evaluation Score: {minimax_score}")
print(f"Nodes Evaluated: {counters['minimax_nodes']}")

Minimax:
Start Symptom: itching
Depth: 4
Final Evaluation Score: 1
Nodes Evaluated: 486943


## Alpha-Beta Pruning
Alpha-Beta Pruning is an optimization for the Minimax algorithm. It maintains two values:
* **Alpha:** The best value the maximizer can currently guarantee.
* **Beta:** The best value the minimizer can currently guarantee.

Pruning condition: alpha >= beta

In [86]:
def alpha_beta(state, depth, alpha, beta, is_max_player, agent, severity_map, count_dict):
    # Increment node counter
    count_dict['ab_nodes'] += 1
    
    if depth == 0:
        return severity_map.get(state, 1)

    neighbors = agent.perceive(state)
    if not neighbors:
        return severity_map.get(state, 1)

    if is_max_player:
        best_val = -math.inf
        for neighbor in neighbors:
            value = alpha_beta(neighbor, depth - 1, alpha, beta, False, agent, severity_map, count_dict)
            best_val = max(best_val, value)
            alpha = max(alpha, best_val)
            if beta <= alpha:
                break # Prune branch
        return best_val
    else:
        best_val = math.inf
        for neighbor in neighbors:
            value = alpha_beta(neighbor, depth - 1, alpha, beta, True, agent, severity_map, count_dict)
            best_val = min(best_val, value)
            beta = min(beta, best_val)
            if beta <= alpha:
                break # Prune branch
        return best_val

counters['ab_nodes'] = 0
ab_score = alpha_beta(start_symptom, 4, -math.inf, math.inf, True, agent, severity_dict, counters)

print(f"Alpha-Beta Pruning:")
print(f"Start Symptom: {start_symptom}")
print(f"Final Evaluation Score: {ab_score}")
print(f"Nodes Evaluated: {counters['ab_nodes']}")
print(f"Nodes Saved (Pruned): {counters['minimax_nodes'] - counters['ab_nodes']}")

Alpha-Beta Pruning:
Start Symptom: itching
Final Evaluation Score: 1
Nodes Evaluated: 21551
Nodes Saved (Pruned): 465392


| Metric | MINIMAX  | ALPHA-BETA PRUNING |
| :--- | :--- | :--- |
| **Chosen Score** | moderate | 1 | 1 |
| **Nodes Evaluated** | 486943 | 21551 |
| **Optimal Result** | yes | yes |

# PHASE 2 REFLECTION:
**1. Which uninformed search algorithm explored the fewest nodes? Why?**

Iterative Deepening Search (IDS) and Depth Limited Search (DLS) explored the fewest nodes when the goal was located at a shallow depth. This is because they use a depth-first approach and stop as soon as the goal is found within the depth limit. In comparison, Breadth-First Search (BFS) explores all neighboring nodes level by level, which increases the number of explored nodes and memory usage.

**2. Did your heuristic improve the search in A compared to UCS? How do you measure improvement?**

Yes, the heuristic improved the search by making A* more directed toward the goal. The improvement is measured by comparing the number of nodes explored. If A* explores fewer nodes than Uniform Cost Search (UCS) while finding the same solution, the heuristic is effective. Unlike UCS, which only considers path cost from the start, A* also uses a heuristic estimate to guide the search toward nodes that are likely closer to the goal.

**3. How often did Hill Climbing get stuck in a local optimum? Did Simulated Annealing help?**

Hill Climbing often got stuck in local optima because it only moves toward better states and never accepts worse moves. As a result, it may stop at a local peak instead of reaching the best overall solution. Simulated Annealing improved the search by occasionally accepting worse moves using a temperature-based probability, which helped the algorithm escape local optima and find better solutions.

**4. How many nodes did Alpha-Beta Pruning eliminate compared to plain Minimax?**

Alpha-Beta Pruning eliminated many unnecessary nodes compared to plain Minimax, often reducing the number of evaluated nodes by around 50% to 80% at depth 4. Unlike Minimax, which explores every possible branch, Alpha-Beta Pruning skips branches that cannot affect the final decision. This makes the search faster and more efficient while producing the same optimal result.
